In [4]:
#!/usr/bin/env python3
"""
rebuild_reactome_csv.py
========================
Rebuilds reactome_interactions_with_pathways.csv using:
  1. Reactome interactions filtered to intercellular only
     (both proteins must be extracellular/membrane per UniProt)
  2. No KEGG — Reactome pathways only
  3. Optional LR pair filtering
"""
from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np
import requests

# ── File paths ────────────────────────────────────────────────────────────────
REACTOME_INTERACTIONS  = Path("reactome_interactions_human.txt")
UNIPROT_PATHWAYS       = Path("uniprot_to_pathways.txt")
UNIPROT_SYM_CSV        = Path("../../og_gene_set/ppi_chloe/d_uniprot_to_symbol_20251120.csv")
UNIPROT_SUBCELL_TSV    = Path("uniprot_subcellular_human.tsv")
MULTIPLEX_DIR          = Path("../multixrank_input_consolidated/multiplex/gene_multiplex")
LR_PAIRS_CSV           = Path("human_lr_pairs.csv")       # set to None to skip
OUTPUT_CSV             = Path("reactome_interactions_with_pathways_v2.csv")

# ── Toggle ────────────────────────────────────────────────────────────────────
INCLUDE_LR_PAIRS       = True    # set False to exclude LR pair filtering
REQUIRE_IN_MULTIPLEX   = True    # set False to keep all intercellular pairs

# ── Step 1 — Load UniProt → gene symbol map ───────────────────────────────────
print("=" * 70)
print("STEP 1 — Load UniProt → gene symbol map")
print("=" * 70)
sym_df = pd.read_csv(UNIPROT_SYM_CSV)
sym_df.columns = sym_df.columns.str.strip().str.lower()
uid_col = next(c for c in sym_df.columns if c in ("uniprotid", "uniprot_id", "uniprot"))
sym_col = next(c for c in sym_df.columns if c in ("symbol", "gene_symbol", "genesymbol"))

uniprot_to_symbol: dict[str, str] = {}
for _, row in sym_df.iterrows():
    uid = str(row[uid_col]).strip().upper()
    sym = str(row[sym_col]).strip().upper()
    if uid and sym and uid != "NAN" and sym != "NAN":
        uniprot_to_symbol[uid] = sym
print(f"  Loaded {len(uniprot_to_symbol):,} UniProt → symbol entries")

# ── Step 2 — Load multiplex gene symbols ─────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 2 — Load multiplex gene symbols from layer TSVs")
print("=" * 70)
multiplex_symbols: set[str] = set()
layer_files = sorted(MULTIPLEX_DIR.glob("*_layer.tsv"))
cell_type_layers = [
    lf for lf in layer_files
    if not any(x in lf.name.lower() for x in ("pathway", "ligand", "receptor"))
]
print(f"  Cell-type layer files found: {len(cell_type_layers):,}")
for lf in cell_type_layers:
    sub = pd.read_csv(
        lf, sep="\t", header=None,
        names=["g1", "g2"], dtype=str, on_bad_lines="skip"
    )
    for col in ("g1", "g2"):
        multiplex_symbols.update(sub[col].dropna().str.strip().str.upper())
multiplex_symbols.discard("NAN")
print(f"  Unique gene symbols across all layers: {len(multiplex_symbols):,}")

# ── Step 3 — Download/load UniProt subcellular locations ─────────────────────
print("\n" + "=" * 70)
print("STEP 3 — Load UniProt subcellular locations")
print("=" * 70)

def download_uniprot_subcellular():
    url = (
        'https://rest.uniprot.org/uniprotkb/stream?'
        'query=organism_id:9606+AND+reviewed:true'
        '&fields=accession,cc_subcellular_location'
        '&format=tsv'
    )
    print("  Downloading from UniProt REST API...")
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    with open(UNIPROT_SUBCELL_TSV, 'w') as f:
        f.write(response.text)
    df = pd.read_csv(UNIPROT_SUBCELL_TSV, sep='\t')
    df.columns = ['uniprot_id', 'subcellular_location']
    return df

if UNIPROT_SUBCELL_TSV.exists():
    print(f"  Loading cached file: {UNIPROT_SUBCELL_TSV}")
    subcell_df = pd.read_csv(UNIPROT_SUBCELL_TSV, sep='\t')
    subcell_df.columns = ['uniprot_id', 'subcellular_location']
else:
    subcell_df = download_uniprot_subcellular()

print(f"  Loaded {len(subcell_df):,} UniProt subcellular entries")

# Build extracellular/membrane protein set
intercellular_terms = [
    'Secreted', 'Extracellular space', 'Extracellular matrix',
    'Cell membrane', 'Plasma membrane'
]
pattern = '|'.join(intercellular_terms)
mask = subcell_df['subcellular_location'].str.contains(pattern, case=False, na=False)
extracellular_proteins = set(subcell_df[mask]['uniprot_id'].str.strip().str.upper())

print(f"  Extracellular/membrane proteins: {len(extracellular_proteins):,}")
for term in intercellular_terms:
    n = subcell_df['subcellular_location'].str.contains(term, case=False, na=False).sum()
    print(f"    '{term}': {n:,}")

# ── Step 4 — Load and filter Reactome interactions ────────────────────────────
print("\n" + "=" * 70)
print("STEP 4 — Load and filter Reactome interactions (intercellular only)")
print("=" * 70)
interactions_df = pd.read_csv(REACTOME_INTERACTIONS, sep="\t")
interactions_df.columns = interactions_df.columns.str.strip()
print(f"  Raw interactions: {len(interactions_df):,} rows")

# Identify UniProt columns
uniprot_cols = [c for c in interactions_df.columns if "uniprot" in c.lower()]
col_uid1 = uniprot_cols[0]
col_uid2 = uniprot_cols[1]

# Clean UniProt IDs
interactions_df["uniprot1"] = (
    interactions_df[col_uid1].astype(str)
    .str.replace("uniprotkb:", "", regex=False)
    .str.strip().str.upper()
)
interactions_df["uniprot2"] = (
    interactions_df[col_uid2].astype(str)
    .str.replace("uniprotkb:", "", regex=False)
    .str.strip().str.upper()
)

# Drop self-interactions
before = len(interactions_df)
interactions_df = interactions_df[
    interactions_df["uniprot1"] != interactions_df["uniprot2"]
].copy()
print(f"  Removed {before - len(interactions_df):,} self-interactions")

# Filter to intercellular: both proteins must be extracellular/membrane
mask_1 = interactions_df["uniprot1"].isin(extracellular_proteins)
mask_2 = interactions_df["uniprot2"].isin(extracellular_proteins)
interactions_filtered = interactions_df[mask_1 & mask_2].copy()

print(f"  Interactor 1 extracellular: {mask_1.sum():,}")
print(f"  Interactor 2 extracellular: {mask_2.sum():,}")
print(f"  Both extracellular:         {len(interactions_filtered):,}")
print(f"  Survival rate:              {100*len(interactions_filtered)/len(interactions_df):.1f}%")

# Map to gene symbols
interactions_filtered["symbol1"] = interactions_filtered["uniprot1"].map(uniprot_to_symbol)
interactions_filtered["symbol2"] = interactions_filtered["uniprot2"].map(uniprot_to_symbol)

both_mapped = interactions_filtered["symbol1"].notna() & interactions_filtered["symbol2"].notna()
interactions_filtered = interactions_filtered[both_mapped].copy()
print(f"  After symbol mapping:       {len(interactions_filtered):,} rows")

# Optionally restrict to multiplex genes
if REQUIRE_IN_MULTIPLEX:
    in_mp = (
        interactions_filtered["symbol1"].isin(multiplex_symbols) &
        interactions_filtered["symbol2"].isin(multiplex_symbols)
    )
    interactions_filtered = interactions_filtered[in_mp].copy()
    print(f"  After multiplex filter:     {len(interactions_filtered):,} rows")

# Spot check known intercellular pairs
known_pairs = [('TGFB1', 'TGFBR1'), ('IL6', 'IL6R'), ('EGF', 'EGFR')]
print("\n  Spot checks:")
for lig, rec in known_pairs:
    found = (
        (
            (interactions_filtered["symbol1"] == lig) &
            (interactions_filtered["symbol2"] == rec)
        ) | (
            (interactions_filtered["symbol1"] == rec) &
            (interactions_filtered["symbol2"] == lig)
        )
    ).any()
    print(f"    {lig} ↔ {rec}: {'✅' if found else '❌ MISSING'}")

# ── Step 5 — Load Reactome UniProt → pathway map ─────────────────────────────
print("\n" + "=" * 70)
print("STEP 5 — Load Reactome pathway map")
print("=" * 70)
up_df = pd.read_csv(UNIPROT_PATHWAYS, sep="\t", header=None)
up_df.columns = ["uniprot_id", "reactome_id", "url",
                  "pathway_name", "evidence_code", "species"]

up_human = up_df[up_df["species"] == "Homo sapiens"].copy()
up_human["uniprot_id"]    = up_human["uniprot_id"].str.strip().str.upper()
up_human["pathway_name"]  = up_human["pathway_name"].str.strip()   # strip whitespace

print(f"  Human Reactome pathway rows: {len(up_human):,}")
print(f"  Unique proteins:             {up_human['uniprot_id'].nunique():,}")
print(f"  Unique pathway names:        {up_human['pathway_name'].nunique():,}")

# Collapse to one row per UniProt ID
reactome_collapsed = (
    up_human
    .groupby("uniprot_id")["pathway_name"]
    .apply(lambda x: "|".join(sorted({p.strip() for p in x})))
    .reset_index()
    .rename(columns={"pathway_name": "reactome_pathways"})
)

# ── Step 6 — Optionally load LR pairs ────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 6 — LR pairs")
print("=" * 70)
lr_pairs_set: set[tuple[str, str]] = set()

if INCLUDE_LR_PAIRS and LR_PAIRS_CSV is not None and Path(LR_PAIRS_CSV).exists():
    lr_df = pd.read_csv(LR_PAIRS_CSV)
    for _, row in lr_df.iterrows():
        lig = str(row.get('ligand_gname',  '')).strip().upper()
        rec = str(row.get('receptor_gname', '')).strip().upper()
        if lig and rec and lig != 'NAN' and rec != 'NAN':
            lr_pairs_set.add((min(lig, rec), max(lig, rec)))
    print(f"  Loaded {len(lr_pairs_set):,} LR pairs")
else:
    print("  LR pairs excluded (INCLUDE_LR_PAIRS=False or file not found)")

# ── Step 7 — Merge pathway info onto interactions ─────────────────────────────
print("\n" + "=" * 70)
print("STEP 7 — Merge pathway info onto interactions")
print("=" * 70)
merged = interactions_filtered.copy()

# Reactome pathways for protein 1
merged = merged.merge(
    reactome_collapsed.rename(columns={
        "uniprot_id": "uid1", "reactome_pathways": "reactome_p1"
    }),
    left_on="uniprot1", right_on="uid1", how="left"
).drop(columns="uid1", errors="ignore")

# Reactome pathways for protein 2
merged = merged.merge(
    reactome_collapsed.rename(columns={
        "uniprot_id": "uid2", "reactome_pathways": "reactome_p2"
    }),
    left_on="uniprot2", right_on="uid2", how="left"
).drop(columns="uid2", errors="ignore")

print(f"  Merged shape: {merged.shape}")

# ── Step 8 — Compute common pathways ─────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 8 — Compute common pathways")
print("=" * 70)

def _split(val) -> set[str]:
    if pd.isna(val) or str(val).strip() in ("", "nan", "NAN"):
        return set()
    return {p.strip() for p in str(val).split("|") if p.strip()}

def compute_common_pathway(row) -> str | None:
    r1 = _split(row.get("reactome_p1"))
    r2 = _split(row.get("reactome_p2"))
    common = r1 & r2
    return "|".join(sorted(common)) if common else None

print("  Computing common pathways...")
merged["common_pathway"] = merged.apply(compute_common_pathway, axis=1)
merged["pathway_source"] = merged["common_pathway"].apply(
    lambda x: "Reactome" if pd.notna(x) else None
)

n_with_pw = merged["common_pathway"].notna().sum()
print(f"  Rows with ≥1 common pathway: {n_with_pw:,} / {len(merged):,} "
      f"({100*n_with_pw/max(len(merged),1):.1f}%)")

# ── Step 9 — Deduplicate protein pairs ───────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 9 — Deduplicate protein pairs")
print("=" * 70)

# Normalise pair order for deduplication
merged["pair_key"] = merged.apply(
    lambda r: tuple(sorted([r["uniprot1"], r["uniprot2"]])), axis=1
)

before = len(merged)
merged_unique = (
    merged
    .groupby("pair_key", sort=False)
    .agg(
        uniprot1=("uniprot1", "first"),
        uniprot2=("uniprot2", "first"),
        symbol1=("symbol1",  "first"),
        symbol2=("symbol2",  "first"),
        common_pathway=("common_pathway",
                        lambda x: "|".join(sorted({
                            p for val in x.dropna()
                            for p in str(val).split("|") if p.strip()
                        })) or None),
        pathway_source=("pathway_source", "first"),
        reactome_p1=("reactome_p1", "first"),
        reactome_p2=("reactome_p2", "first"),
        n_reaction_contexts=("pair_key", "count"),
    )
    .reset_index(drop=True)
)

# Fix empty string → None after aggregation
merged_unique["common_pathway"] = merged_unique["common_pathway"].replace("", None)

print(f"  Before dedup: {before:,} rows")
print(f"  After dedup:  {len(merged_unique):,} unique protein pairs")
print(f"  Average reaction contexts per pair: {before/max(len(merged_unique),1):.1f}")

# ── Step 10 — Flag LR pairs ───────────────────────────────────────────────────
if lr_pairs_set:
    merged_unique["is_lr_pair"] = merged_unique.apply(
        lambda r: (
            min(r["symbol1"], r["symbol2"]),
            max(r["symbol1"], r["symbol2"])
        ) in lr_pairs_set,
        axis=1
    )
    print(f"\n  Rows flagged as LR pairs: {merged_unique['is_lr_pair'].sum():,}")
else:
    merged_unique["is_lr_pair"] = False

# ── Step 11 — Drop self-pairs and save ───────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 11 — Save")
print("=" * 70)

before = len(merged_unique)
merged_unique = merged_unique[
    merged_unique["symbol1"] != merged_unique["symbol2"]
].copy()
print(f"  Dropped {before - len(merged_unique):,} symbol self-pairs")

keep_cols = [
    "uniprot1", "uniprot2",
    "symbol1", "symbol2",
    "common_pathway", "pathway_source",
    "reactome_p1", "reactome_p2",
    "n_reaction_contexts", "is_lr_pair",
]
keep_cols = [c for c in keep_cols if c in merged_unique.columns]
out_df = merged_unique[keep_cols].copy()

out_df.to_csv(OUTPUT_CSV, index=False)
print(f"  Saved → {OUTPUT_CSV} ({len(out_df):,} rows)")

# Quick summary
from collections import Counter
print(f"\n  Unique pairs with named pathway: "
      f"{out_df['common_pathway'].notna().sum():,}")
print(f"\n  Top 20 shared pathways:")
counter = Counter(
    p for val in out_df['common_pathway'].dropna()
    for p in str(val).split("|") if p.strip()
)
for pw, n in counter.most_common(20):
    print(f"    {n:>6,}  {pw}")

STEP 1 — Load UniProt → gene symbol map
  Loaded 165,580 UniProt → symbol entries

STEP 2 — Load multiplex gene symbols from layer TSVs
  Cell-type layer files found: 64
  Unique gene symbols across all layers: 17,051

STEP 3 — Load UniProt subcellular locations
  Loading cached file: uniprot_subcellular_human.tsv
  Loaded 20,431 UniProt subcellular entries
  Extracellular/membrane proteins: 5,766
    'Secreted': 2,147
    'Extracellular space': 360
    'Extracellular matrix': 280
    'Cell membrane': 4,045
    'Plasma membrane': 493

STEP 4 — Load and filter Reactome interactions (intercellular only)
  Raw interactions: 123,895 rows
  Removed 7,771 self-interactions
  Interactor 1 extracellular: 34,253
  Interactor 2 extracellular: 31,372
  Both extracellular:         16,332
  Survival rate:              14.1%
  After symbol mapping:       16,150 rows
  After multiplex filter:     14,745 rows

  Spot checks:
    TGFB1 ↔ TGFBR1: ✅
    IL6 ↔ IL6R: ✅
    EGF ↔ EGFR: ❌ MISSING

STEP 5 — L